# Mapa semántico de tesis del CIDE

Este cuaderno construye un mapa semántico de los resúmenes y títulos de las tesis de Licenciatura en Economía usando embeddings multilingües con `tencent/KaLM-Embedding-Gemma3-12B-2511` y reducción de dimensionalidad con UMAP.


## Dependencias necesarias

Ejecuta la siguiente celda si trabajas en un ambiente nuevo. Instala `sentence-transformers` (para el modelo `tencent/KaLM-Embedding-Gemma3-12B-2511`), `umap-learn`, `scikit-learn` y `plotly` para visualización interactiva.


In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
import torch
import umap
from sklearn.cluster import KMeans
import networkx as nx

import plotly.express as px
from IPython.display import display


## Cargar y preparar los datos

Ajusta `DATA_PATH` si guardaste los datos en otra ruta. Se combinan título y resumen para crear el texto que se enviará al modelo de embeddings.


In [2]:
df_raw = pd.read_csv("tesis_lic_economia_cide.csv")

# Filtrar a idioma español si existe la columna
#if 'idioma' in df_raw.columns:
#    before = len(df_raw)
#    df_raw = df_raw[df_raw['idioma'].str.lower().str.startswith('spa', na=False)]
#    print(f"Filtrado por idioma='spa': {len(df_raw)} filas (antes {before}).")

# Normalizamos texto y eliminamos filas sin información útil
text_cols = ['titulo', 'resumen']
df = df_raw.dropna(subset=text_cols).copy()
df['texto'] = (df['titulo'].fillna('') + '. ' + df['resumen'].fillna('')).str.strip()
df = df[df['texto'].str.len() > 20].reset_index(drop=True)

print(f'Tesis disponibles para analizar: {len(df)}')
display(df.head(3))

Tesis disponibles para analizar: 462


,año_pub,autorx,titulo,resumen,asesor,asesor_unificado,idioma,tipo_acceso,url,texto
0,1998,"Ballinez Ambriz, Roberto",Inestabilidad política e inversión: México 198...,El episodio mexicano a lo largo de 1989 hasta ...,NaN,NaN,spa,Acceso abierto,https://repositorio-digital.cide.edu/handle/11...,Inestabilidad política e inversión: México 198...
1,2004,"Escobar Montecinos, Ninel Geraldine",Determinantes del gasto de agua para uso domés...,El estudio que aquí se presenta es de carácter...,Juan Manuel Torres Rojo,Dr. Juan Manuel Torres Rojo,spa,Acceso abierto,https://repositorio-digital.cide.edu/handle/11...,Determinantes del gasto de agua para uso domés...
2,2004,"Gómez Gasteasoro, Irene",Estabilizadores fiscales autómaticos: un análi...,El objetivo central de este trabajo es hacer u...,Francisco Alejandro Villagómez Amezcua,Dr. Alejandro Villagómez Amezcua,spa,Acceso abierto,https://repositorio-digital.cide.edu/handle/11...,Estabilizadores fiscales autómaticos: un análi...


## Generar embeddings con `hiiamsid/sentence_similarity_spanish_es`


In [3]:
model_name = 'hiiamsid/sentence_similarity_spanish_es'
model = SentenceTransformer(model_name)

textos = df['texto'].tolist()
embeddings = model.encode(
    textos,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print('Embeddings generados:', embeddings.shape)


Batches: 100%|██████████| 15/15 [00:31<00:00,  2.13s/it]


Embeddings generados: (462, 768)



## Reducir dimensionalidad con UMAP


In [4]:

umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.05,
    metric='cosine',
    random_state=420,
)

umap_2d = umap_model.fit_transform(embeddings)

df_layout = df.copy()
df_layout['umap_x'] = umap_2d[:, 0]
df_layout['umap_y'] = umap_2d[:, 1]
df_layout.head(3)


,año_pub,autorx,titulo,resumen,asesor,asesor_unificado,idioma,tipo_acceso,url,texto,umap_x,umap_y
0,1998,"Ballinez Ambriz, Roberto",Inestabilidad política e inversión: México 198...,El episodio mexicano a lo largo de 1989 hasta ...,NaN,NaN,spa,Acceso abierto,https://repositorio-digital.cide.edu/handle/11...,Inestabilidad política e inversión: México 198...,8.320200,6.030946
1,2004,"Escobar Montecinos, Ninel Geraldine",Determinantes del gasto de agua para uso domés...,El estudio que aquí se presenta es de carácter...,Juan Manuel Torres Rojo,Dr. Juan Manuel Torres Rojo,spa,Acceso abierto,https://repositorio-digital.cide.edu/handle/11...,Determinantes del gasto de agua para uso domés...,7.199337,7.872823
2,2004,"Gómez Gasteasoro, Irene",Estabilizadores fiscales autómaticos: un análi...,El objetivo central de este trabajo es hacer u...,Francisco Alejandro Villagómez Amezcua,Dr. Alejandro Villagómez Amezcua,spa,Acceso abierto,https://repositorio-digital.cide.edu/handle/11...,Estabilizadores fiscales autómaticos: un análi...,8.063041,6.415943



## Clustering con K-Means


In [5]:
max_clusters = 11
n_clusters = min(max_clusters, len(df_layout))
if n_clusters < 2:
    raise ValueError('Se necesitan al menos dos tesis para ejecutar K-Means.')

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=11,
)

# K-Means se entrena sobre el layout 2D de UMAP
cluster_labels = kmeans.fit_predict(umap_2d)
cluster_algo = 'kmeans (umap_2d)'

labels_series = pd.Series(cluster_labels, index=df_layout.index)

df_layout['cluster_id'] = labels_series
df_layout['cluster_label'] = labels_series.map(lambda x: f'Cluster {x}')

print(f'Algoritmo de clustering activo: {cluster_algo} (k={n_clusters})')

cluster_summary = (
    df_layout.groupby('cluster_label')
    .agg(
        conteo=('titulo', 'count'),
        año_promedio=('año_pub', 'mean'),
    )
    .sort_values('conteo', ascending=False)
)

display(cluster_summary)

Algoritmo de clustering activo: kmeans (umap_2d) (k=11)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


,conteo,año_promedio
cluster_label,,
Cluster 9,48,2019.041667
Cluster 1,46,2021.195652
Cluster 6,46,2015.673913
Cluster 10,45,2009.044444
Cluster 2,42,2012.190476
Cluster 4,42,2011.928571
Cluster 7,42,2015.595238
Cluster 0,41,2013.292683
Cluster 3,38,2015.052632


In [6]:

hover_cols = {
    'titulo': True,
    'autorx': True,
    'año_pub': True,
    'asesor_unificado': True,
    'url': True,
    'cluster_label': False,
}

fig = px.scatter(
    df_layout,
    x='umap_x',
    y='umap_y',
    color='cluster_label',
    hover_name='titulo',
    hover_data=hover_cols,
    title='Mapa semántico de tesis de Lic. en Economía (CIDE)',
    template='plotly_white',
    width=950,
    height=650,
)

fig.update_traces(marker=dict(size=8, opacity=0.82, line=dict(width=0)))
fig.update_layout(
    legend_title_text='Cluster',
    coloraxis_showscale=False,
)

fig.show()


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [7]:
# Top 5 asesores por cluster (barras)
advisor_col = 'asesor_unificado'

advisors_df = df_layout.copy()
advisors_df[advisor_col] = advisors_df[advisor_col].fillna('Asesor desconocido').replace('', 'Asesor desconocido')

for cid in sorted(advisors_df['cluster_id'].unique()):
    cluster_name = f'Cluster {cid}' if cid != -1 else 'Ruido'
    subset = advisors_df[advisors_df['cluster_id'] == cid]

    top5 = (
        subset.groupby(advisor_col)
        .size()
        .reset_index(name='conteo')
        .sort_values('conteo', ascending=False)
        .head(5)
    )

    if top5.empty:
        print(f'Sin datos de asesores para {cluster_name}.')
        continue

    fig = px.bar(
        top5,
        x='conteo',
        y=advisor_col,
        orientation='h',
        color=advisor_col,
        title=f'Top 5 asesores en {cluster_name}',
        text='conteo',
        color_discrete_sequence=px.colors.qualitative.Plotly,
    )

    fig.update_traces(textposition='outside')
    fig.update_layout(
        template='plotly_white',
        height=320,
        xaxis_title='Número de tesis',
        yaxis_title='Asesor/a',
        yaxis=dict(categoryorder='array', categoryarray=top5[advisor_col].tolist()[::-1]),
        showlegend=False,
        margin=dict(t=60, b=40, l=40, r=20),
    )

    fig.show()

In [8]:
import xlsxwriter

# Exportar a XLSX: tesis agrupadas por cluster
output_path = Path('clusters_tesis.xlsx')

cols_export = [
    'cluster_label',
    'cluster_id',
    'titulo',
    'resumen',
    'autorx',
    'año_pub',
    'asesor_unificado',
    'url',
]

# Resumen por cluster
cluster_overview = (
    df_layout.groupby(['cluster_label', 'cluster_id'])
    .size()
    .reset_index(name='conteo')
    .sort_values('conteo', ascending=False)
)

with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    # Hoja de resumen
    cluster_overview.to_excel(writer, sheet_name='Resumen', index=False)

    # Una hoja por cluster con títulos y resúmenes
    for cid in sorted(df_layout['cluster_id'].unique()):
        cluster_name = f'Cluster {cid}' if cid != -1 else 'Ruido'
        sheet_name = cluster_name[:31]  # límite de Excel

        subset = df_layout[df_layout['cluster_id'] == cid].copy()
        subset = subset[cols_export]
        subset.to_excel(writer, sheet_name=sheet_name, index=False)

print(f'Archivo exportado: {output_path.resolve()}')

Archivo exportado: /Volumes/Files/R/ScrapUMAP_TesisLicEcoCIDE/clusters_tesis.xlsx


In [9]:
# Grafo no dirigido por asesores: comunidades de tesis que comparten asesor
from itertools import combinations

advisor_col = 'asesor_unificado'

graph_df = df_layout.copy()
graph_df[advisor_col] = graph_df[advisor_col].fillna('Asesor desconocido').replace('', 'Asesor desconocido')

G = nx.Graph()
G.add_nodes_from(graph_df.index)

# Conectar tesis que tienen el mismo asesor
for advisor, idxs in graph_df.groupby(advisor_col).groups.items():
    ids = list(idxs)
    if len(ids) < 2:
        continue
    for i, j in combinations(ids, 2):
        w = G.get_edge_data(i, j, {}).get('weight', 0) + 1
        G.add_edge(i, j, weight=w, advisor=advisor)

if G.number_of_edges() == 0:
    print('No hay conexiones por asesor (todos únicos o faltantes).')
    df_layout['advisor_comm_id'] = -1
    df_layout['advisor_comm_label'] = 'Sin comunidad'
else:
    communities = list(nx.algorithms.community.greedy_modularity_communities(G, weight='weight'))
    comm_map = {node: cid for cid, nodes in enumerate(communities) for node in nodes}

    df_layout['advisor_comm_id'] = df_layout.index.map(lambda x: comm_map.get(x, -1))
    df_layout['advisor_comm_label'] = df_layout['advisor_comm_id'].map(lambda x: f'AdvisorComm {x}' if x != -1 else 'Sin comunidad')

    comm_summary = (
        df_layout.groupby('advisor_comm_label')
        .agg(
            conteo=('titulo', 'count'),
            asesores_top=('asesor_unificado', lambda s: ', '.join(s.value_counts().head(3).index)),
        )
        .sort_values('conteo', ascending=False)
    )

    print(f'Comunidades detectadas: {len(communities)}')
    display(comm_summary.head(20))

Comunidades detectadas: 77


,conteo,asesores_top
advisor_comm_label,,
AdvisorComm 0,37,Dr. Fausto Hernández Trillo
AdvisorComm 1,24,Dr. Rodolfo Cermeño Bazán
AdvisorComm 2,23,Dr. Víctor Gerardo Carreón Rodríguez
AdvisorComm 6,22,Dr. Kurt Unger Rubín
AdvisorComm 5,22,Dr. Daniel Ventosa-Santaulària
AdvisorComm 4,22,Dr. David Arie Mayer Foulkes
AdvisorComm 3,22,Dr. Alejandro Villagómez Amezcua
AdvisorComm 7,20,Dr. Gustavo A. del Ángel Mobarak
AdvisorComm 8,17,Dra. Eva Olimpia Arceo Gómez


In [10]:
# Visualizar la red de asesores (no dirigido)
import plotly.graph_objects as go

if 'advisor_comm_label' not in df_layout.columns or G.number_of_edges() == 0:
    print('No hay red de asesores para visualizar.')
else:
    pos = nx.spring_layout(G, weight='weight', seed=42)

    nodes_df = (
        df_layout[['titulo', 'asesor_unificado', 'cluster_label', 'advisor_comm_label']]
        .assign(
            x=lambda d: d.index.map(lambda i: pos[i][0]),
            y=lambda d: d.index.map(lambda i: pos[i][1]),
        )
    )

    communities = sorted(nodes_df['advisor_comm_label'].unique())
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Safe + px.colors.qualitative.Set3
    color_map = {c: palette[i % len(palette)] for i, c in enumerate(communities)}
    nodes_df['color'] = nodes_df['advisor_comm_label'].map(color_map)

    edge_x, edge_y = [], []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        line=dict(width=0.5, color='#999'),
        hoverinfo='none',
        mode='lines'
    )

    node_trace = go.Scatter(
        x=nodes_df['x'],
        y=nodes_df['y'],
        mode='markers',
        marker=dict(
            size=8,
            color=nodes_df['color'],
            line=dict(width=0.5, color='#333'),
        ),
        text=nodes_df.apply(
            lambda r: f"{r['titulo']}<br>Asesor: {r['asesor_unificado']}<br>Comunidad: {r['advisor_comm_label']}<br>Cluster UMAP: {r['cluster_label']}",
            axis=1,
        ),
        hoverinfo='text',
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title='Red de tesis conectadas por asesor (comunidades por modularidad)',
        template='plotly_white',
        showlegend=False,
        width=950,
        height=700,
        margin=dict(l=30, r=30, t=60, b=30),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
    )

    fig.show()